# 🔍 IEEE-CIS Fraud Detection — Exploratory Data Analysis

**Project:** Self-Healing MLOps Fraud Detection Pipeline  
**Week:** 1 of 6  
**Goal:** Understand the data before we model it. In real MLOps, 70% of your time is here.

---

## What are we looking at?

The **IEEE-CIS Fraud Detection** dataset from Kaggle contains:
- ~590,000 e-commerce transactions
- 434 features (yes, that many!)
- Binary target: `isFraud` (1 = fraudulent, 0 = legitimate)
- Only ~3.5% of transactions are fraud → **heavily imbalanced dataset**

### The two files:
| File | Contents |
|------|----------|
| `train_transaction.csv` | Payment data: amount, card info, device, timestamps |
| `train_identity.csv` | Identity data: browser, OS, device type, IP |

They join on `TransactionID`.

In [ ]:
# ── Imports ──────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Make plots look nice
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')
pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', '{:.3f}'.format)

print('✅ Libraries loaded')

In [ ]:
# ── Load Data ─────────────────────────────────────────────
# If running in Colab, mount Drive first and update these paths
TRANSACTION_PATH = '../data/raw/train_transaction.csv'
IDENTITY_PATH = '../data/raw/train_identity.csv'

df_trans = pd.read_csv(TRANSACTION_PATH, low_memory=False)
df_id = pd.read_csv(IDENTITY_PATH, low_memory=False)

print(f'Transactions: {df_trans.shape}')
print(f'Identity:     {df_id.shape}')

# Merge on TransactionID
df = df_trans.merge(df_id, on='TransactionID', how='left')
print(f'Merged:       {df.shape}')

## 1. Target Distribution — The Imbalance Problem

This is the FIRST thing you always check in fraud detection.  
If 96.5% of data is "not fraud", a dumb model that says "never fraud" gets 96.5% accuracy.  
That's why we use **AUC-ROC** instead of accuracy.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Count plot
fraud_counts = df['isFraud'].value_counts()
axes[0].bar(['Legitimate (0)', 'Fraud (1)'], fraud_counts.values, 
            color=['#2ecc71', '#e74c3c'], edgecolor='black', linewidth=0.5)
axes[0].set_title('Transaction Count by Class', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Count')
for i, v in enumerate(fraud_counts.values):
    axes[0].text(i, v + 1000, f'{v:,}', ha='center', fontweight='bold')

# Pie chart
axes[1].pie(fraud_counts.values, labels=['Legitimate', 'Fraud'],
            autopct='%1.2f%%', colors=['#2ecc71', '#e74c3c'],
            startangle=90, explode=(0, 0.1))
axes[1].set_title('Class Distribution', fontsize=13, fontweight='bold')

plt.suptitle('⚠️  Severely Imbalanced Dataset — Use AUC-ROC, not Accuracy!', 
             fontsize=11, color='red', y=1.02)
plt.tight_layout()
plt.savefig('../docs/01_class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'\nFraud rate: {df["isFraud"].mean()*100:.2f}%')
print(f'Imbalance ratio: {fraud_counts[0]/fraud_counts[1]:.1f}:1')

## 2. Transaction Amount Analysis

Fraudulent transactions often have different amount patterns than legitimate ones.  
Let's check if `TransactionAmt` is a strong signal.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

fraud = df[df['isFraud'] == 1]['TransactionAmt']
legit = df[df['isFraud'] == 0]['TransactionAmt']

# Distribution of raw amounts
axes[0].hist(legit.clip(upper=1000), bins=50, alpha=0.6, color='#2ecc71', label='Legit', density=True)
axes[0].hist(fraud.clip(upper=1000), bins=50, alpha=0.6, color='#e74c3c', label='Fraud', density=True)
axes[0].set_title('Amount Distribution (clipped at $1000)')
axes[0].set_xlabel('Transaction Amount ($)')
axes[0].legend()

# Log-transformed amounts
axes[1].hist(np.log1p(legit), bins=50, alpha=0.6, color='#2ecc71', label='Legit', density=True)
axes[1].hist(np.log1p(fraud), bins=50, alpha=0.6, color='#e74c3c', label='Fraud', density=True)
axes[1].set_title('Log(Amount) Distribution')
axes[1].set_xlabel('log(1 + TransactionAmt)')
axes[1].legend()

# Box plot comparison
df_plot = df[['TransactionAmt', 'isFraud']].copy()
df_plot['TransactionAmt_log'] = np.log1p(df_plot['TransactionAmt'])
df_plot['Class'] = df_plot['isFraud'].map({0: 'Legitimate', 1: 'Fraud'})
df_plot.boxplot(column='TransactionAmt_log', by='Class', ax=axes[2])
axes[2].set_title('Log Amount by Class')
axes[2].set_xlabel('')

plt.tight_layout()
plt.savefig('../docs/02_amount_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nAmount Statistics:')
print(df.groupby('isFraud')['TransactionAmt'].describe().round(2))

## 3. Time Patterns — When Does Fraud Happen?

`TransactionDT` is seconds elapsed from some reference point (not a real timestamp).  
We extract hour-of-day and day-of-week patterns.

In [ ]:
df['hour'] = (df['TransactionDT'] // 3600) % 24
df['day_of_week'] = (df['TransactionDT'] // (3600 * 24)) % 7

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Fraud rate by hour
hourly = df.groupby('hour')['isFraud'].agg(['mean', 'count']).reset_index()
axes[0].bar(hourly['hour'], hourly['mean'] * 100, color='#e74c3c', alpha=0.8)
axes[0].set_title('Fraud Rate by Hour of Day', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Hour (0 = midnight)')
axes[0].set_ylabel('Fraud Rate (%)')
axes[0].axhline(df['isFraud'].mean() * 100, color='black', linestyle='--', 
                label=f'Average ({df["isFraud"].mean()*100:.1f}%)')
axes[0].legend()

# Transaction volume by hour
ax2 = axes[0].twinx()
ax2.plot(hourly['hour'], hourly['count'], color='#3498db', linewidth=2, label='Volume')
ax2.set_ylabel('Transaction Count', color='#3498db')

# Fraud rate by day
daily = df.groupby('day_of_week')['isFraud'].mean().reset_index()
days = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']
axes[1].bar(range(7), daily['isFraud'] * 100, color='#9b59b6', alpha=0.8)
axes[1].set_xticks(range(7))
axes[1].set_xticklabels(days)
axes[1].set_title('Fraud Rate by Day of Week', fontsize=13, fontweight='bold')
axes[1].set_ylabel('Fraud Rate (%)')
axes[1].axhline(df['isFraud'].mean() * 100, color='black', linestyle='--')

plt.tight_layout()
plt.savefig('../docs/03_time_patterns.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Peak fraud hour: {hourly.loc[hourly["mean"].idxmax(), "hour"]}:00')

## 4. Missing Values — The Biggest Data Quality Issue

Many of the 434 features are heavily missing. This is intentional — Vesta (the data provider) 
anonymized and obscured many columns. We need to know WHICH ones are missing to decide 
how to handle them.

In [ ]:
# Calculate missing percentage per column
missing = (df.isnull().sum() / len(df) * 100).sort_values(ascending=False)
missing = missing[missing > 0]

print(f'Columns with missing values: {len(missing)}')
print(f'Columns with >50% missing:   {(missing > 50).sum()}')
print(f'Columns with >90% missing:   {(missing > 90).sum()}')

# Plot top 40 most missing
fig, ax = plt.subplots(figsize=(14, 8))
missing_top40 = missing.head(40)
colors = ['#e74c3c' if x > 50 else '#f39c12' if x > 20 else '#2ecc71' 
          for x in missing_top40.values]

bars = ax.barh(range(len(missing_top40)), missing_top40.values, color=colors)
ax.set_yticks(range(len(missing_top40)))
ax.set_yticklabels(missing_top40.index, fontsize=8)
ax.set_xlabel('Missing %')
ax.set_title('Top 40 Features by Missing Value %', fontsize=13, fontweight='bold')
ax.axvline(50, color='red', linestyle='--', label='>50% missing (consider dropping)')
ax.axvline(20, color='orange', linestyle='--', label='>20% missing')
ax.legend()
ax.invert_yaxis()

plt.tight_layout()
plt.savefig('../docs/04_missing_values.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Top Predictive Features — What Signals Fraud?

We'll check which features have the strongest statistical difference  
between fraud and non-fraud transactions.

In [ ]:
# Check fraud rates for card-level features
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Card type fraud rates
if 'card4' in df.columns:
    card4_fraud = df.groupby('card4')['isFraud'].agg(['mean', 'count'])
    card4_fraud = card4_fraud[card4_fraud['count'] > 100].sort_values('mean', ascending=False)
    axes[0].bar(card4_fraud.index, card4_fraud['mean'] * 100, 
                color='#e74c3c', alpha=0.8, edgecolor='black')
    axes[0].set_title('Fraud Rate by Card Network', fontsize=12, fontweight='bold')
    axes[0].set_ylabel('Fraud Rate (%)')
    axes[0].axhline(df['isFraud'].mean() * 100, color='black', 
                    linestyle='--', label='Overall average')
    axes[0].legend()

# Product type fraud rates  
if 'ProductCD' in df.columns:
    prod_fraud = df.groupby('ProductCD')['isFraud'].agg(['mean', 'count'])
    prod_fraud = prod_fraud.sort_values('mean', ascending=False)
    axes[1].bar(prod_fraud.index, prod_fraud['mean'] * 100,
                color='#9b59b6', alpha=0.8, edgecolor='black')
    axes[1].set_title('Fraud Rate by Product Category', fontsize=12, fontweight='bold')
    axes[1].set_ylabel('Fraud Rate (%)')
    axes[1].axhline(df['isFraud'].mean() * 100, color='black', linestyle='--')

plt.tight_layout()
plt.savefig('../docs/05_feature_fraud_rates.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Correlation Heatmap — Key Numeric Features

In [ ]:
# Select key numeric features for correlation analysis
key_cols = ['isFraud', 'TransactionAmt', 'hour', 'day_of_week',
            'card1', 'card2', 'card3', 'card5', 'addr1', 'addr2',
            'dist1', 'dist2', 'C1', 'C2', 'C3', 'C4', 'C5',
            'C6', 'C7', 'C8', 'C9', 'C10']

available = [c for c in key_cols if c in df.columns]
corr = df[available].corr()

fig, ax = plt.subplots(figsize=(14, 10))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', 
            cmap='RdYlGn', center=0, ax=ax,
            annot_kws={'size': 8}, linewidths=0.5)
ax.set_title('Correlation Matrix — Key Features', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../docs/06_correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

# Show top correlations with isFraud
print('\nTop correlations with isFraud:')
print(corr['isFraud'].abs().sort_values(ascending=False).head(15))

## 7. EDA Summary & Takeaways

| Insight | Implication for Model |
|---------|----------------------|
| 3.5% fraud rate | Use `scale_pos_weight=28` in XGBoost; evaluate with AUC-ROC not accuracy |
| Fraud spikes at certain hours | `Transaction_hour` is an important feature |
| Log(Amount) is normally distributed | Use `log1p(TransactionAmt)` as a feature |
| Many features >50% missing | Fill with -999 sentinel; XGBoost handles this natively |
| Card network affects fraud rate | `card4` is a useful categorical feature |
| Product type affects fraud rate | `ProductCD` is a useful categorical feature |

**Next step:** `src/train.py` — train the XGBoost baseline model using these insights.

In [ ]:
# Save a clean version of key stats for our README
summary = {
    'total_transactions': len(df),
    'fraud_count': int(df['isFraud'].sum()),
    'fraud_rate_pct': round(df['isFraud'].mean() * 100, 2),
    'total_features': df.shape[1],
    'missing_columns': int((df.isnull().sum() > 0).sum()),
    'avg_transaction_amt': round(df['TransactionAmt'].mean(), 2),
}

import json
with open('../docs/eda_summary.json', 'w') as f:
    json.dump(summary, f, indent=2)

print('📊 EDA Summary:')
for k, v in summary.items():
    print(f'   {k}: {v}')

print('\n✅ EDA complete! Ready to train in src/train.py')